# G1 · Capa GOLD · Rendimiento de cultivos por municipio y año

**Grupo 1** · Ruta agroambiental · Pipeline E12 → producto E13 · Responsable: Julián

**Pregunta del producto:** ¿qué cultivos tienen mejor rendimiento por municipio y año?

Este notebook implementa la capa GOLD del diagrama `Diagrama ejercicio 12.excalidraw`. Lee `silver/unificado/` (salida de Brayan), calcula en DuckDB el rendimiento de cada cultivo por municipio y año como **cociente de sumas** (t/ha), lo compara con el rendimiento nacional del mismo cultivo (`indice_relativo`), ordena los cultivos de cada municipio-año y comprueba que producción y área cuadran con SILVER. El plan de la capa está en [`03_GOLD_Julian.md`](../plan_grupo_1/03_GOLD_Julian.md).

| Paso del diagrama | Sección |
|---|---|
| Verificación de fuentes ya almacenadas (antiduplicidad) | 2 y 5.3 |
| Carga de los datos pre-procesados en el motor | 3.1 |
| Agrupamiento y generación de información relevante | 3.2 |
| Almacenamiento | 3.3, 4 y 5 |

**Entrada** (`ENTRADA_DIR`): carpeta `silver/`; por ahora el fixture `soluciones/plan_grupo_1/fixtures/silver`.
**Salida** (`kit/salidas/grupo_1/<run_id>/gold/`): `rendimiento_cultivo.parquet`, `conciliacion.parquet`, `consultas_gold.sql` y `controles_gold.json`, más una línea en `kit/salidas/grupo_1/registro_gold.jsonl`.

**Cómo ejecutar.** Con el kernel del `.venv` del curso, *Kernel → Restart & Run All*. Si la entrada ya se procesó, la sección 2 detiene el notebook sin escribir resultados; con `FORZAR = True` se reprocesa.

> **Origen.** Notebook convertido del script `G1_03_gold_py.md` de Julián, **sin ejecutar**. El código es el original repartido en celdas (los títulos de sección del script pasan a ser títulos Markdown). Solo cambian cuatro líneas que no funcionan dentro de un notebook; están marcadas con `# Adaptación al convertir`:
>
> | Original | En el notebook | Motivo |
> |---|---|---|
> | `from soluciones.E12_solucion import ...` | Copia de las tres funciones | `E12_solucion` es un `.ipynb` y no se puede importar |
> | `encontrar_repositorio(nombre_archivo_marcador="fuentes.json")` | `encontrar_repositorio()` | La función no tiene ese parámetro; la copia ya busca `kit/fuentes.json` |
> | `exit(0)` | `raise SystemExit(0)` | En Jupyter `exit()` apaga el kernel |
> | `firma_dataframe(df_gold)` | `firma_dataframe(df_gold, columnas, orden)` | La función de E12 exige columnas y orden; se ordena por la clave única |

## 1. Rutas y configuración del repositorio

La primera celda carga las librerías y tres funciones de `soluciones/E12_solucion.ipynb`:

- `encontrar_repositorio`: sube por las carpetas hasta encontrar `kit/fuentes.json` y devuelve la raíz del repositorio.
- `sha256_archivo`: huella SHA-256 de un archivo, leída por bloques.
- `firma_dataframe`: huella del **contenido** de una tabla (ordena filas, convierte a texto y calcula SHA-256).

La segunda define las rutas:

- `ENTRADA_DIR`: carpeta `silver/`. Hasta el 4 nov, el fixture; después, `kit/salidas/grupo_1/<run_id>/silver` de Brayan.
- `SALIDA_DIR`: carpeta nueva `kit/salidas/grupo_1/<RUN_ID>/gold/`; `RUN_ID` son 12 caracteres aleatorios en cada ejecución.
- `REGISTRO_PATH`: historial de ejecuciones de GOLD, una línea JSON por ejecución válida.
- `FORZAR`: `True` reprocesa aunque la entrada ya tenga una ejecución válida.

In [ ]:
import json
import uuid
import duckdb
import hashlib
from pathlib import Path
# Adaptación al convertir: soluciones/E12_solucion es un notebook (.ipynb), no un módulo, así que
# `from soluciones.E12_solucion import ...` falla. Se copian aquí las tres funciones
# (secciones 1 y 2 de E12_solucion.ipynb), con kit/fuentes.json como marcador del repositorio.


def encontrar_repositorio(inicio=None):
    inicio = Path.cwd() if inicio is None else Path(inicio)
    for candidato in (inicio, *inicio.parents):
        if (candidato / 'kit' / 'fuentes.json').is_file():
            return candidato
        anidado = candidato / 'big-data-postgraduate'
        if (anidado / 'kit' / 'fuentes.json').is_file():
            return anidado
    raise FileNotFoundError('No se encontró la raíz de big-data-postgraduate')


def sha256_archivo(ruta, bloque=1024 * 1024):
    firma = hashlib.sha256()
    with Path(ruta).open('rb') as archivo:
        for fragmento in iter(lambda: archivo.read(bloque), b''):
            firma.update(fragmento)
    return firma.hexdigest()


def firma_dataframe(datos, columnas, orden):
    normalizado = datos.loc[:, columnas].copy()
    normalizado = normalizado.sort_values(orden, kind='stable').reset_index(drop=True)
    normalizado = normalizado.astype('string').fillna('<NA>')
    texto = normalizado.to_csv(index=False, lineterminator='\n')
    return hashlib.sha256(texto.encode('utf-8')).hexdigest()

In [ ]:
# Adaptación al convertir: la función no tiene el parámetro nombre_archivo_marcador; la copia ya busca kit/fuentes.json.
REPOSITORIO = encontrar_repositorio()

RUN_ID = uuid.uuid4().hex[:12]
# Entrada temporal hasta el 4 de nov (fixtures). Cambiar a kit/salidas/grupo_1/<run_id>/silver para producción
ENTRADA_DIR = REPOSITORIO / 'soluciones' / 'plan_grupo_1' / 'fixtures' / 'silver'  
SALIDA_DIR = REPOSITORIO / 'kit' / 'salidas' / 'grupo_1' / RUN_ID / 'gold'
REGISTRO_PATH = REPOSITORIO / 'kit' / 'salidas' / 'grupo_1' / 'registro_gold.jsonl'
FORZAR = False

SALIDA_DIR.mkdir(parents=True, exist_ok=True)
REGISTRO_PATH.parent.mkdir(parents=True, exist_ok=True)

## 2. Antiduplicidad e idempotencia (firma de entrada)

`firma_entrada` es el SHA-256 de las huellas de todos los Parquet de `ENTRADA_DIR`, recorridos en orden de ruta. Si `registro_gold.jsonl` ya tiene una ejecución `valida` con la misma firma y `FORZAR = False`, la celda muestra dónde quedó esa salida y detiene el notebook sin recalcular.

In [ ]:
archivos_silver = sorted(list(ENTRADA_DIR.glob("**/*.parquet")))
firmas_archivos = "".join([sha256_archivo(p) for p in archivos_silver])
firma_entrada = hashlib.sha256(firmas_archivos.encode('utf-8')).hexdigest()

ya_procesado = False
if REGISTRO_PATH.exists() and not FORZAR:
    with open(REGISTRO_PATH, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                reg = json.loads(line)
                if reg.get("firma_entrada") == firma_entrada and reg.get("estado") == "valida":
                    print(f"Ya procesado anteriormente en: {reg.get('ruta_salida')}")
                    ya_procesado = True
                    break

if ya_procesado and not FORZAR:
    print("Ejecución omitida por antiduplicidad.")
    # Adaptación al convertir: en Jupyter exit() apaga el kernel; SystemExit detiene "Run All" sin apagarlo.
    raise SystemExit(0)

## 3. Consultas en DuckDB

### 3.1 Carga de `unificado`

Crea la vista `unificado` sobre el Parquet Hive de SILVER (`anio=…/codigo_departamento=…`). `hive_types` fija `anio` como entero y `codigo_departamento` como texto, para que `05` no se convierta en `5`.

In [ ]:
con = duckdb.connect()

# Carga de vista unificado con lectura Hive
con.execute(f"""
CREATE OR REPLACE VIEW unificado AS
SELECT * FROM read_parquet('{ENTRADA_DIR.as_posix()}/unificado/**/*.parquet',
    hive_partitioning = true,
    hive_types = {{'anio': 'BIGINT', 'codigo_departamento': 'VARCHAR'}});
""")

### 3.2 Rendimiento, índice relativo y ranking

La consulta tiene cuatro pasos:

1. `base_apta`: solo filas con `apto_rendimiento = true` (área cosechada > 0 y producción ≥ 0), para no dividir por cero.
2. `agregado_municipio`: por municipio × año × cultivo × estado físico suma producción (t) y área cosechada (ha y m²) y calcula `rendimiento_t_ha = Σ producción / Σ área`, un cociente de sumas y no un promedio de rendimientos. `rendimiento_kg_m2 = rendimiento_t_ha / 10`, porque 1 t/ha = 1 000 kg / 10 000 m². `muestras_suelo` y `mediana_ph` tienen un solo valor por municipio-año; `MAX` lo conserva.
3. `agregado_nacional`: el mismo cociente agrupando solo por año, cultivo y estado físico (todo el país).
4. `rendimiento_completo`: `indice_relativo = rendimiento del municipio / rendimiento nacional` (1,3 = 30 % por encima del país) y `rank_municipio_anio` con `DENSE_RANK` descendente por índice dentro de cada municipio-año; 1 = mejor.

In [ ]:
query_gold = """
WITH base_apta AS (
    SELECT *
    FROM unificado
    WHERE apto_rendimiento = true
),
agregado_municipio AS (
    SELECT 
        codigo_departamento,
        departamento,
        codigo_municipio,
        municipio,
        anio,
        cultivo,
        estado_fisico,
        SUM(produccion_t) AS produccion_t,
        SUM(area_cosechada_ha) AS area_cosechada_ha,
        SUM(area_cosechada_m2) AS area_cosechada_m2,
        SUM(produccion_t) / SUM(area_cosechada_ha) AS rendimiento_t_ha,
        (SUM(produccion_t) / SUM(area_cosechada_ha)) / 10.0 AS rendimiento_kg_m2,
        COUNT(*) AS n_registros,
        MAX(muestras_suelo) AS muestras_suelo,
        MAX(mediana_ph) AS mediana_ph
    FROM base_apta
    GROUP BY codigo_departamento, departamento, codigo_municipio, municipio, anio, cultivo, estado_fisico
),
agregado_nacional AS (
    SELECT 
        anio,
        cultivo,
        estado_fisico,
        SUM(produccion_t) / SUM(area_cosechada_ha) AS rendimiento_nacional_t_ha
    FROM base_apta
    GROUP BY anio, cultivo, estado_fisico
),
rendimiento_completo AS (
    SELECT 
        m.*,
        n.rendimiento_nacional_t_ha,
        (m.rendimiento_t_ha / n.rendimiento_nacional_t_ha) AS indice_relativo,
        DENSE_RANK() OVER (
            PARTITION BY m.codigo_municipio, m.anio 
            ORDER BY (m.rendimiento_t_ha / n.rendimiento_nacional_t_ha) DESC
        ) AS rank_municipio_anio
    FROM agregado_municipio m
    JOIN agregado_nacional n 
      ON m.anio = n.anio 
     AND m.cultivo = n.cultivo 
     AND m.estado_fisico = n.estado_fisico
)
SELECT 
    codigo_departamento, departamento, codigo_municipio, municipio,
    anio, cultivo, estado_fisico, produccion_t, area_cosechada_ha,
    area_cosechada_m2, rendimiento_t_ha, rendimiento_kg_m2,
    rendimiento_nacional_t_ha, indice_relativo, rank_municipio_anio,
    n_registros, muestras_suelo, mediana_ph
FROM rendimiento_completo
ORDER BY codigo_municipio, anio, rank_municipio_anio;
"""

### 3.3 Cálculo y escritura de `rendimiento_cultivo.parquet`

Una fila por municipio × año × cultivo × estado físico, en el orden `codigo_municipio, anio, rank_municipio_anio`.

In [ ]:
df_gold = con.execute(query_gold).df()
ruta_rendimiento = SALIDA_DIR / "rendimiento_cultivo.parquet"
df_gold.to_parquet(ruta_rendimiento, index=False)

## 4. Tabla de conciliación (SILVER vs GOLD)

Por municipio-año compara producción (t) y área (ha) de las filas aptas de `unificado` con las de `rendimiento_cultivo`. `cuadra` es verdadero si ambas diferencias son menores que 1e-6. La fila `total` suma los grupos y solo cuadra si cuadran todos. Se guarda en `conciliacion.parquet`.

In [ ]:
query_conciliacion = f"""
WITH silver_totales AS (
    SELECT 
        codigo_municipio,
        anio,
        SUM(produccion_t) AS produccion_silver_t,
        SUM(area_cosechada_ha) AS area_silver_ha
    FROM unificado
    WHERE apto_rendimiento = true
    GROUP BY codigo_municipio, anio
),
gold_totales AS (
    SELECT 
        codigo_municipio,
        anio,
        SUM(produccion_t) AS produccion_gold_t,
        SUM(area_cosechada_ha) AS area_gold_ha
    FROM read_parquet('{ruta_rendimiento.as_posix()}')
    GROUP BY codigo_municipio, anio
),
grupos AS (
    SELECT 
        'grupo' AS nivel,
        s.codigo_municipio,
        s.anio,
        s.produccion_silver_t,
        g.produccion_gold_t,
        s.area_silver_ha,
        g.area_gold_ha,
        (s.produccion_silver_t - g.produccion_gold_t) AS diferencia_produccion_t,
        (s.area_silver_ha - g.area_gold_ha) AS diferencia_area_ha,
        (ABS(s.produccion_silver_t - g.produccion_gold_t) < 1e-6 AND ABS(s.area_silver_ha - g.area_gold_ha) < 1e-6) AS cuadra
    FROM silver_totales s
    JOIN gold_totales g ON s.codigo_municipio = g.codigo_municipio AND s.anio = g.anio
),
total AS (
    SELECT 
        'total' AS nivel,
        NULL AS codigo_municipio,
        NULL AS anio,
        SUM(produccion_silver_t) AS produccion_silver_t,
        SUM(produccion_gold_t) AS produccion_gold_t,
        SUM(area_silver_ha) AS area_silver_ha,
        SUM(area_gold_ha) AS area_gold_ha,
        SUM(diferencia_produccion_t) AS diferencia_produccion_t,
        SUM(diferencia_area_ha) AS diferencia_area_ha,
        (SUM(CASE WHEN cuadra THEN 1 ELSE 0 END) = COUNT(*)) AS cuadra
    FROM grupos
)
SELECT * FROM grupos
UNION ALL
SELECT * FROM total;
"""

df_conciliacion = con.execute(query_conciliacion).df()
ruta_conciliacion = SALIDA_DIR / "conciliacion.parquet"
df_conciliacion.to_parquet(ruta_conciliacion, index=False)

## 5. Archivos auxiliares y controles

### 5.1 `consultas_gold.sql`

Guarda las consultas de rendimiento y de conciliación para que Juan y el equipo auditor las repitan.

In [ ]:
with open(SALIDA_DIR / "consultas_gold.sql", "w", encoding="utf-8") as f:
    f.write("-- Consulta 1: Creación de vista unificado\n")
    f.write("-- Consulta 2: Cálculo de rendimiento_cultivo con índice relativo y rank\n")
    f.write(query_gold)
    f.write("\n-- Consulta 3: Conciliación de datos Silver vs Gold\n")
    f.write(query_conciliacion)

### 5.2 Controles del contrato

Si un `assert` falla, el notebook se detiene y no se escriben `controles_gold.json` ni la línea del registro.

- Clave `(codigo_municipio, anio, cultivo, estado_fisico)` única.
- Sin rendimientos nulos ni negativos.
- `rendimiento_kg_m2 = rendimiento_t_ha / 10` en todas las filas.
- Conciliación con `cuadra = true` en todas las filas, incluida `total`.
- Cada municipio tiene al menos un cultivo con `rank_municipio_anio = 1`.

In [ ]:
# Validaciones estrictas del contrato
duplicados = df_gold.duplicated(subset=['codigo_municipio', 'anio', 'cultivo', 'estado_fisico']).sum()
assert duplicados == 0, "Error: Claves duplicadas en rendimiento_cultivo"
assert df_gold['rendimiento_t_ha'].isna().sum() == 0, "Error: Rendimientos nulos detectados"
assert (df_gold['rendimiento_t_ha'] < 0).sum() == 0, "Error: Rendimientos negativos detectados"
assert ((df_gold['rendimiento_t_ha'] / 10.0 - df_gold['rendimiento_kg_m2']).abs() < 1e-6).all(), "Error en conversión kg/m2"
assert df_conciliacion['cuadra'].all(), "Error: La conciliación no cuadró al 100%"

ranks_1 = df_gold[df_gold['rank_municipio_anio'] == 1]
assert len(ranks_1['codigo_municipio'].unique()) == len(df_gold['codigo_municipio'].unique()), "Error: Faltan ranks = 1"

print("¡Todos los controles pasaron correctamente!")

### 5.3 `controles_gold.json` y registro de la ejecución

Resume la ejecución (filas, grupos conciliados, producción total en t, área total en ha y `firma_contenido` de `rendimiento_cultivo`) en `controles_gold.json` y agrega la misma línea a `registro_gold.jsonl`. La sección 2 usa ese registro en la siguiente ejecución para detectar duplicados.

In [ ]:
controles = {
    "run_id": RUN_ID,
    "firma_entrada": firma_entrada,
    "estado": "valida",
    "filas_unificado": int(con.execute("SELECT COUNT(*) FROM unificado").fetchone()[0]),
    "filas_aptas": int(con.execute("SELECT COUNT(*) FROM unificado WHERE apto_rendimiento = true").fetchone()[0]),
    "filas_gold": len(df_gold),
    "grupos_conciliados": len(df_conciliacion[df_conciliacion['nivel'] == 'grupo']),
    "grupos_con_diferencia": int((~df_conciliacion['cuadra']).sum()),
    "produccion_total_t": float(df_gold['produccion_t'].sum()),
    "area_total_ha": float(df_gold['area_cosechada_ha'].sum()),
    # Adaptación al convertir: firma_dataframe de E12 exige columnas y orden; se ordena por la clave única.
    "firma_contenido": firma_dataframe(df_gold, list(df_gold.columns), ['codigo_municipio', 'anio', 'cultivo', 'estado_fisico']),
    "ruta_salida": str(SALIDA_DIR),
    "errores": []
}

with open(SALIDA_DIR / "controles_gold.json", "w", encoding="utf-8") as f:
    json.dump(controles, f, indent=2)

with open(REGISTRO_PATH, "a", encoding="utf-8") as f:
    f.write(json.dumps(controles) + "\n")